## ✂️ How the Recursive Text Splitter Works

The **Recursive Text Splitter** is an intelligent approach used to divide extensive texts into smaller segments, known as **chunks**. The core idea is to recursively break down the text while respecting defined size limits and, simultaneously, attempting to preserve the semantic integrity of the snippets. Instead of a simple linear division, this method searches for natural separation points—such as paragraph breaks or sentences—to prevent the loss of vital context.



### The Logic Behind Recursive Splitting

The operational flow of this algorithm involves the following steps:

1.  **Size Verification:** The algorithm evaluates whether the text fits within a predetermined limit (e.g., 600 characters). If it does, there is no need for further division.
2.  **Identification of Split Points:** If the text is too long, the method looks for natural delimiters—such as spaces, punctuation, or line breaks—to find a split point that doesn't interrupt a complete idea.
3.  **Applying Overlap:** To prevent relevant information from being lost at the boundary between two chunks, a **sliding window** or **overlap** (e.g., 150 characters) is added between the generated pieces. This technique helps maintain contextual continuity and improves retrieval accuracy.
4.  **Recursive Division:** If the initial attempt at a split point doesn't produce a chunk of the appropriate size, the process is applied recursively to the resulting segments until all pieces are within the desired limits.



### Advantages and Challenges

**Key Benefits:**
* **Context Preservation:** Crucial for systems relying on semantic retrieval, such as **RAG pipelines**.
* **Flexibility:** Allows for the customization of sizes and overlaps based on the nature of the content.

**Main Challenges:**
* **Parameter Tuning:** Choosing the ideal `chunk_size` and `chunk_overlap` often requires testing and iteration for different document types.
* **Computational Complexity:** Recursive splitting can increase processing time for extremely large documents, requiring optimization to maintain performance.

### Practical Code Example

Below is a simplified example of how a Recursive Text Splitter can be implemented in Python:

```python
def recursive_text_splitter(text, max_length=600, overlap=150):
    # If text is already small enough, return it as a single chunk
    if len(text) <= max_length:
        return [text]

    # Attempt to find a preferred split point (like a space)
    break_point = text.rfind(' ', 0, max_length)
    if break_point == -1:
        break_point = max_length

    # Define the current chunk
    chunk = text[:break_point]

    # Define the remaining text, applying the overlap
    # We move back by 'overlap' characters to maintain context
    remainder = text[max(0, break_point - overlap):]

    # Recurse on the remainder
    return [chunk] + recursive_text_splitter(remainder, max_length, overlap)

# Example usage
document = "A long text that needs to be split into several pieces intelligently..."
chunks = recursive_text_splitter(document)

for i, c in enumerate(chunks):
    print(f"Chunk {i+1}: {c}\n")
```

This example illustrates the basic logic of how to split text recursively, ensuring that no segment exceeds the defined size while maintaining part of the context between the generated pieces.

This approach is valuable in scenarios where precision in information retrieval is essential, allowing **LLM** and **RAG-based systems** to obtain well-grounded answers from extensive documents.

### PROJETO PRÁTICO 2

In [1]:
#!pip install langchain langchain-google-genai pypdf langchain-community

import os
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_classic.embeddings import OpenAIEmbeddings
from langchain_classic.vectorstores import Chroma
from langchain_classic.chat_models import ChatOpenAI
from langchain_classic.chains import RetrievalQA

In [2]:
caminhos_bulas = ["dipirona.pdf", "paracetamol.pdf"]
documentos = []

for caminho in caminhos_bulas:
    loader = PyPDFLoader(f"../data/{caminho}")
    docs = loader.load()

    for doc in docs:
        doc.metadata["medicamento"] = caminho.split("/")[-1].replace(".pdf", "")

    documentos.extend(docs)

len(documentos)

11

In [3]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=150)

chunks = text_splitter.split_documents(documentos)

len(chunks)

61

In [4]:
# 4. Enrichment with metadata
for chunk in chunks:
    texto = chunk.page_content.lower()

    if "identificação do medicamento" in texto or "composição" in texto:
        chunk.metadata["categoria"] = "identificacao"

    elif "indicação" in texto or "para que este medicamento é indicado" in texto:
        chunk.metadata["categoria"] = "indicacao"

    elif "como este medicamento funciona" in texto or "ação" in texto:
        chunk.metadata["categoria"] = "como_funciona"

    elif "contraindicação" in texto or "quando não devo usar" in texto:
        chunk.metadata["categoria"] = "contraindicacao"

    elif "advertência" in texto or "precaução" in texto or "o que devo saber antes de usar" in texto:
        chunk.metadata["categoria"] = "advertencias_precaucoes"

    elif "interação" in texto or "interações medicamentosas" in texto:
        chunk.metadata["categoria"] = "interacoes"

    elif "dose" in texto or "posologia" in texto or "como devo usar" in texto:
        chunk.metadata["categoria"] = "posologia_modo_uso"

    elif "reações adversas" in texto or "quais os males" in texto:
        chunk.metadata["categoria"] = "reacoes_adversas"

    elif "onde, como e por quanto tempo posso guardar" in texto or "armazenar" in texto:
        chunk.metadata["categoria"] = "armazenamento"

    elif "quantidade maior do que a indicada" in texto or "superdosagem" in texto:
        chunk.metadata["categoria"] = "superdosagem"

    else:
        chunk.metadata["categoria"] = "geral"

In [5]:
import random

chunks_aleatorios = random.sample(chunks, 2)

for i, chunk in enumerate(chunks_aleatorios, start=1):
    print(
        f"\n--- Chunk {i} ---\n"
        f"Metadados:\n"
        f"{chunk.metadata}\n"
        f"\nConteúdo (início)"
        f"{chunk.page_content[:300]}"
    )


--- Chunk 1 ---
Metadados:
{'producer': 'PDFium', 'creator': 'PDFium', 'creationdate': 'D:20191206103835', 'source': '../data/dipirona.pdf', 'total_pages': 10, 'page': 4, 'page_label': '5', 'medicamento': 'dipirona', 'categoria': 'como_funciona'}

Conteúdo (início)ácido acetilsalicílico:  a dipirona pode reduzir o efeito do ácido acetilsalicílico na agregação plaquetária 
(união das plaquetas que atuam na coagulação), quando usados concomitan temente. Portanto, essa 
combinação deve ser usada com precaução em pacientes que tomam baixa dose de ácido acetilsali

--- Chunk 2 ---
Metadados:
{'producer': 'Adobe PDF Library 15.0', 'creator': 'Adobe InDesign CC 13.0 (Windows)', 'creationdate': '2017-12-21T09:26:33-02:00', 'gts_pdfxconformance': 'PDF/X-1a:2001', 'gts_pdfxversion': 'PDF/X-1:2001', 'moddate': '2017-12-21T09:26:33-02:00', 'title': 'PARACETAMOL.indd', 'trapped': '/False', 'source': '../data/paracetamol.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'medicamento': 'paraceta

In [6]:
# 5. Embedding generation and vector store
from langchain_google_genai import GoogleGenerativeAIEmbeddings

from langchain_community.vectorstores import Chroma

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001"
)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    persist_directory="./chroma_bulas"
)

In [7]:
# 6. Context retriever

retriever = vectorstore.as_retriever(
    search_kwargs={"k":4}
)

In [8]:
# 7. RAG pipeline integration

from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash"
)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [9]:
# 8. Test and validation

pergunta = "Quais são as contraindicações do dipirona?"

resposta = qa_chain.invoke(pergunta)

print(f"Pergunta:\n{pergunta}\n")
print(f"Resposta:\n{resposta["result"]}\n")

print("\nTrechos utilizados como contexto:")
for i, doc in enumerate(resposta["source_documents"], start=1):
    print(f"--- Trecho {i} ---")

    # Metadados principais
    print(f"Medicamento: {doc.metadata.get('medicamento', 'N/A')}")
    print(f"Categoria: {doc.metadata.get('categoria', 'N/A')}")
    print(f"Documento: {doc.metadata.get('source', 'Documento desconhecido')}")
    print(f"Página: {doc.metadata.get('page', 'N/A')}")

    # Conteúdo recuperado
    print("\nConteúdo do chunk:")
    print(doc.page_content)
    print("\n")

Pergunta:
Quais são as contraindicações do dipirona?

Resposta:
A dipirona monoidratada não deve ser utilizada caso você tenha:

*   Alergia ou intolerância à dipirona ou a qualquer um dos componentes da formulação.
*   Alergia ou intolerância a outras pirazolonas ou a pirazolidinas (exemplos: fenazona, propifenazona, isopropilaminofenazona, fenilbutazona, oxifembutazona).
*   Experiência prévia de agranulocitose (diminuição acentuada na contagem de glóbulos brancos do sangue) com alguma dessas substâncias.


Trechos utilizados como contexto:
--- Trecho 1 ---
Medicamento: dipirona
Categoria: como_funciona
Documento: ../data/dipirona.pdf
Página: 2

Conteúdo do chunk:
palidez. 
 
Choque an afilático: (reação alérgica grave): ocorre principalmente em pacientes sensíveis. Portanto, a 
dipirona deve ser usada com cautela em pacientes que apresentem alergia atópica ou asma (vide “Quando 
não devo usar este medicamento?”). 
 
Reações cutâneas graves:  foram relatadas reações cutâneas graves c

# 📚 Lesson Recap: RAG & Chunking Strategies

In this lesson, we covered:

* **The Importance of Chunking:** How to divide long documents into smaller, more efficient parts.
* **Impact of Chunk Size:** How the size of chunks affects the inclusion of essential information and document retrieval performance.
* **Chunking Strategies:** Applying different splitting methods, such as:
    * Character-based splitting
    * Sentence-based splitting
    * Paragraph-based splitting
* **Overlapping Technique:** Using overlaps to maintain contextual continuity across chunks.
* **Metadata Usage:** Labeling chunks with metadata to enhance filtering and organization.
* **RAG Conversational Agent:** Building a practical agent for interpreting pharmaceutical drug labels.
* **LangChain Integration:** Managing the workflow for reading, splitting, and retrieving data.
* **OpenAI API:** Utilizing it for generating embeddings and creating a robust retrieval system.